In [ ]:
from numpy.random.mtrand import f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, PrecisionRecallDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay
from sklearn.metrics import roc_curve, RocCurveDisplay


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

data = pd.DataFrame({
    # Product info
    "product_id": np.arange(1, n+1),

    # Biomulch quality (higher is better)
    "biomulch_cfu_per_gram": np.random.uniform(1e7, 1e9, n),

    # Contamination (lower is better)
    "contamination_cfu_per_gram": np.random.uniform(0, 1e6, n),

    # Farmer behavior
    "awareness": np.random.choice([0, 1], n, p=[0.5, 0.5]),
    "previous_pesticide_use": np.random.choice([0, 1], n),

    # Field conditions
    "pest_severity": np.random.randint(1, 11, n),
    "yield_loss": np.random.uniform(5, 50, n),
})

# 🔥 Create realistic adoption logic

score = (
    0.000000001 * data["biomulch_cfu_per_gram"]   # better product → more adoption
    - 0.000001 * data["contamination_cfu_per_gram"]  # contamination → reduces trust
    + 2 * data["awareness"]                        # awareness is VERY important
    + 1 * data["previous_pesticide_use"]           # used to pest control
    + 0.5 * data["pest_severity"]                  # more pests → more need
    + 0.3 * data["yield_loss"]                     # higher losses → more adoption
)

# Convert to probability-like behavior
threshold = np.percentile(score, 60)
data["adoption"] = (score > threshold).astype(int)

# Add small randomness (real-world noise)
noise = np.random.rand(n)
data.loc[noise > 0.9, "adoption"] = 1 - data["adoption"]

print(data.head())

In [ ]:
data.head()

In [ ]:
data.info()


In [ ]:
corr=data.corr()
corr

In [ ]:
list(data.columns)

In [ ]:
corr_ = corr['biomulch_cfu_per_gram'].sort_values(ascending=False)
print(corr_)

In [ ]:
X=data.drop(columns=['product_id','adoption'])
y=data['adoption']


In [ ]:
X.shape
y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Scaler=StandardScaler()
Scaler.fit(X_train)
X_train_scaled=Scaler.transform(X_train)
X_test_scaled=Scaler.transform(X_test)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
corr_ = corr['biomulch_cfu_per_gram'].sort_values(ascending=False)
print(corr_)


In [ ]:
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, y_prob)
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

importance = model.feature_importances_

plt.barh(X.columns, importance)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

In [ ]:
model=RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

In [ ]:
print(PrecisionRecallDisplay.from_predictions(y_test, y_prob))
print(RocCurveDisplay.from_predictions(y_test, y_prob))
plt.show()

In [ ]:
importance = model.feature_importances_

plt.barh(X.columns, importance)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

In [ ]:
X['contamination_cfu_per_gram']